# FY1_3 IRISeq full pipeline reproduction

This notebook reproduces the FY1_3 / Female Young 1 Section 2 IRISeq workflow from FASTQ files through two author-style outputs:

1. gene-expression UMAP colored by GEO `Annotation`
2. bead-bead interaction spatial reconstruction UMAP colored by GEO `Annotation`

The processed GEO matrix, metadata, and connection file are used only in the explicit validation section.


## 1. Configuration and imports

The paths and parameters below are intentionally explicit so the notebook can be rerun after the FASTQ and GEO processed files are placed in the input directory.


In [3]:
from pathlib import Path
import os
import sys
import gzip
import json
import shutil
import hashlib
import importlib
import subprocess
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', None)   # 单元格内容不截断
pd.set_option('display.max_columns', None)    # 显示所有列
pd.set_option('display.width', None)          # 自动适配宽度
pd.set_option('display.max_rows', 200)        # 需要时显示更多行

PROJECT_DIR = Path('/p1/zulab_users/jtian/my_jupyter/IRI-seq')
SCRIPT_FROM_GITHUB = PROJECT_DIR / 'script_from_github'
INPUT_DIR = Path('/p2/zulab/jtian/data/IRISeq/demo_FY1_3/data')
OUTPUT_DIR = Path('/p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline')
REFERENCE_DIR = Path('/p2/zulab/jtian/data/IRISeq/reference')

CDNA_SAMPLE = '20231221_FY1_3_cDNA'
BEAD_SAMPLE = '20231221_FY1_3_connection'
SAMPLE_ID = 'FY1_3'

FASTQ_SOURCE_FILES = {
    CDNA_SAMPLE: {
        'run_accession': 'SRR29481311',
        'description': 'Female Young 1 Section 2 cDNA',
        'R1': INPUT_DIR / 'SRR29481311_1.fastq.gz',
        'R2': INPUT_DIR / 'SRR29481311_2.fastq.gz',
    },
    BEAD_SAMPLE: {
        'run_accession': 'SRR29481287',
        'description': 'Female Young 1 Section 2 bead interaction',
        'R1': INPUT_DIR / 'SRR29481287_1.fastq.gz',
        'R2': INPUT_DIR / 'SRR29481287_2.fastq.gz',
        # The author's bead-interaction script reads sample.R3.fastq.gz.
        # The paired SRA/ENA FASTQ layout provides this sender-bead read as _2.
        'R3': INPUT_DIR / 'SRR29481287_2.fastq.gz',
    },
}

PROCESSED_METADATA_CSV = INPUT_DIR / 'GSE270383_meta_data.csv.gz'
PROCESSED_CONNECTION_CSV = INPUT_DIR / 'GSM8341580_20231221_FY1_3_connection.spatial.csv.gz'
PROCESSED_COUNT_MTX = INPUT_DIR / 'GSE270383_count.mtx.gz'
PROCESSED_BARCODES_TSV = INPUT_DIR / 'GSE270383_barcodes.tsv.gz'
PROCESSED_GENES_TSV = INPUT_DIR / 'GSE270383_genes.tsv.gz'

PICKLE_DIR = SCRIPT_FROM_GITHUB / 'Bead_bc_white_list'
BARCODE_1_FILE = PICKLE_DIR / 'Spatial_R2_barcode_1.pickle'
BARCODE_2_FILE = PICKLE_DIR / 'Spatial_R2_barcode_2.pickle'
BARCODE_3_FILE = PICKLE_DIR / 'Spatial_R2_barcode_3.pickle'
BARCODE_4_FILE = PICKLE_DIR / 'Spatial_R2_barcode_4.pickle'
BARCODE_4_BEAD1_FILE = PICKLE_DIR / 'Spatial_R2_barcode_4_bead1.pickle'

STAR_INDEX = REFERENCE_DIR / 'STAR_mm10_GENCODE_M25'
GENE_REFERENCE_PICKLE = REFERENCE_DIR / 'Gene_annotation' / 'mm10_GENCODE_M25_Gene_reference.pickle'

STAR = shutil.which('STAR') or 'STAR'
SAMTOOLS = shutil.which('samtools') or 'samtools'
CUTADAPT = shutil.which('cutadapt') or 'cutadapt'

# Runtime controls. Keep force flags False for resumable reruns.
RUN_CDNA_PIPELINE = True
RUN_BEAD_PIPELINE = True
RUN_EXPRESSION_ANALYSIS = True
RUN_RECONSTRUCTION = True
RUN_VALIDATION = True
FORCE_RELINK_FASTQ = False
FORCE_RERUN_CDNA = False
FORCE_RERUN_BEADS = False
FORCE_RERUN_RECONSTRUCTION = False

CORE_CDNA = 2
CORE_BEADS = 10
UMI_LIMIT_FOR_ADATA = 50
EXPRESSION_MIN_COUNTS = 400
HVG_N_TOP_GENES = 10000
EXPRESSION_PCS = 25
EXPRESSION_NEIGHBOR_PCS = 20
EXPRESSION_UMAP_MIN_DIST = 0.1

# Author CPU reconstruction notebook: log10(n_umi) >= 0.9, equivalent to n_umi >= 8.
MIN_CONNECTION_UMIS = 8
RECONSTRUCTION_MODE = 'auto'  # auto uses dense PCA when the matrix is modest, sparse SVD otherwise.
DENSE_MATRIX_MAX_CELLS = 50_000_000
RECONSTRUCTION_COMPONENTS = 600
RECONSTRUCTION_RANDOM_STATE = 42
RECONSTRUCTION_UMAP_N_NEIGHBORS = 19
RECONSTRUCTION_UMAP_MIN_DIST = 0.23
RECONSTRUCTION_UMAP_SPREAD = 0.5
RECONSTRUCTION_UMAP_EPOCHS = 500

SAMPLE_SHEET_DIR = OUTPUT_DIR / 'sample_sheets'
FASTQ_DIR = OUTPUT_DIR / 'prepared_fastq'
REPORT_DIR = OUTPUT_DIR / 'reports'
FIGURE_OUTPUT = OUTPUT_DIR / 'figures'
VALIDATION_OUTPUT = OUTPUT_DIR / 'validation'

CDNA_OUTPUT = OUTPUT_DIR / 'cDNA'
FASTQ_BARCODE_ATTACHED = CDNA_OUTPUT / 'Fastq_barcode_attached'
FASTQ_TRIMMED = CDNA_OUTPUT / 'Fastq_trimmed'
SAM_STAR = CDNA_OUTPUT / 'Sam_STAR'
SAM_FILTERED = CDNA_OUTPUT / 'Sam_filtered'
SAM_RMDUP = CDNA_OUTPUT / 'Sam_rmdup'
BED_GENE_COUNT = CDNA_OUTPUT / 'Bed_gene_count'
SUMMARY_GENE_COUNT = CDNA_OUTPUT / 'Summary_gene_count'
ADATA_FOLDER = CDNA_OUTPUT / 'Adata'
EXPRESSION_OUTPUT = CDNA_OUTPUT / 'expression_analysis'

BEAD_OUTPUT = OUTPUT_DIR / 'beads'
UMI_ATTACH = BEAD_OUTPUT / 'UMI_attach'
SPATIAL_BARCODE = BEAD_OUTPUT / 'Spatial_barcode_extraction'
DEDUPLICATE_SPATIAL = BEAD_OUTPUT / 'Spatial_barcode_rmdup'
BEAD_REPORT_FOLDER = BEAD_OUTPUT / 'report' / 'read_num_spatial_barcode'
CONNECTION_OUTPUT = OUTPUT_DIR / 'connections'

for directory in [
    OUTPUT_DIR, SAMPLE_SHEET_DIR, FASTQ_DIR, REPORT_DIR, FIGURE_OUTPUT, VALIDATION_OUTPUT,
    CDNA_OUTPUT, FASTQ_BARCODE_ATTACHED, FASTQ_TRIMMED, SAM_STAR, SAM_FILTERED, SAM_RMDUP,
    BED_GENE_COUNT, SUMMARY_GENE_COUNT, ADATA_FOLDER, EXPRESSION_OUTPUT,
    BEAD_OUTPUT, UMI_ATTACH, SPATIAL_BARCODE, DEDUPLICATE_SPATIAL, BEAD_REPORT_FOLDER,
    CONNECTION_OUTPUT,
]:
    directory.mkdir(parents=True, exist_ok=True)

print('INPUT_DIR  =', INPUT_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('STAR       =', STAR)
print('SAMTOOLS   =', SAMTOOLS)
print('CUTADAPT   =', CUTADAPT)


INPUT_DIR  = /p2/zulab/jtian/data/IRISeq/demo_FY1_3/data
OUTPUT_DIR = /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline
STAR       = /p1/zulab_users/jtian/anaconda3/envs/my_IRISeq_py38/bin/STAR
SAMTOOLS   = /p1/zulab_users/jtian/anaconda3/envs/my_IRISeq_py38/bin/samtools
CUTADAPT   = /p1/zulab_users/jtian/anaconda3/envs/my_IRISeq_py38/bin/cutadapt


## 2. Preflight checks

This cell verifies all expected inputs, author scripts, barcode whitelist files, and external command-line tools before starting the expensive steps.


In [5]:
def assert_file(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing {label}: {path}')
    return path

required_files = []
for sample, reads in FASTQ_SOURCE_FILES.items():
    for read_name, path in reads.items():
        if read_name in {'R1', 'R2'}:
            required_files.append((path, f'{sample} {read_name} FASTQ'))

required_files.extend([
    (PROCESSED_METADATA_CSV, 'GEO metadata'),
    (PROCESSED_CONNECTION_CSV, 'GEO processed FY1_3 connection file'),
    (PROCESSED_COUNT_MTX, 'GEO processed count matrix'),
    (PROCESSED_BARCODES_TSV, 'GEO processed barcodes'),
    (PROCESSED_GENES_TSV, 'GEO processed genes'),
    (BARCODE_1_FILE, 'barcode whitelist 1'),
    (BARCODE_2_FILE, 'barcode whitelist 2'),
    (BARCODE_3_FILE, 'barcode whitelist 3'),
    (BARCODE_4_FILE, 'barcode whitelist 4'),
    (BARCODE_4_BEAD1_FILE, 'barcode whitelist 4 bead1'),
    (STAR_INDEX, 'STAR mm10 index directory'),
    (GENE_REFERENCE_PICKLE, 'mm10 gene reference pickle'),
    (SCRIPT_FROM_GITHUB / 'EasySpatial' / 'Spatial_UMI_barcode_extraction.py', 'author cDNA barcode script'),
    (SCRIPT_FROM_GITHUB / 'Bead_interaction_pipeline' / 'UMI_barcode_extraction.py', 'author bead UMI script'),
    (SCRIPT_FROM_GITHUB / 'Sample_reconstruction_code_CPU.ipynb', 'author CPU reconstruction notebook'),
])

preflight_rows = []
for path, label in required_files:
    exists = Path(path).exists()
    preflight_rows.append({'label': label, 'path': str(path), 'exists': exists})
    if not exists:
        raise FileNotFoundError(f'Missing {label}: {path}')

if RUN_CDNA_PIPELINE:
    missing_tools = [tool for tool, resolved in {'STAR': STAR, 'samtools': SAMTOOLS, 'cutadapt': CUTADAPT}.items() if shutil.which(resolved) is None and not Path(resolved).exists()]
    if missing_tools:
        raise EnvironmentError(
            'Missing external tools required for full cDNA processing: ' + ', '.join(missing_tools) +
            '. Add them to PATH or set STAR/SAMTOOLS/CUTADAPT to absolute paths in the configuration cell.'
        )


python_package_checks = {
    'scanpy': RUN_EXPRESSION_ANALYSIS or RUN_VALIDATION,
    'umap': RUN_RECONSTRUCTION,
    'sklearn': RUN_RECONSTRUCTION or RUN_VALIDATION,
    'scipy': RUN_RECONSTRUCTION or RUN_VALIDATION,
    'matplotlib': True,
}
missing_packages = [pkg for pkg, needed in python_package_checks.items() if needed and importlib.util.find_spec(pkg) is None]
if missing_packages:
    raise ImportError(
        'Missing Python packages required by enabled sections: ' + ', '.join(missing_packages) +
        '. Install them in the active Jupyter kernel or set the relevant RUN_* flags to False.'
    )

display(pd.DataFrame(preflight_rows))
print('Preflight checks passed.')


,label,path,exists
0,20231221_FY1_3_cDNA R1 FASTQ,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/data/SRR29481311_1.fastq.gz,True
1,20231221_FY1_3_cDNA R2 FASTQ,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/data/SRR29481311_2.fastq.gz,True
2,20231221_FY1_3_connection R1 FASTQ,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/data/SRR29481287_1.fastq.gz,True
3,20231221_FY1_3_connection R2 FASTQ,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/data/SRR29481287_2.fastq.gz,True
4,GEO metadata,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/data/GSE270383_meta_data.csv.gz,True
5,GEO processed FY1_3 connection file,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/data/GSM8341580_20231221_FY1_3_connection.spatial.csv.gz,True
6,GEO processed count matrix,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/data/GSE270383_count.mtx.gz,True
7,GEO processed barcodes,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/data/GSE270383_barcodes.tsv.gz,True
8,GEO processed genes,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/data/GSE270383_genes.tsv.gz,True
9,barcode whitelist 1,/p1/zulab_users/jtian/my_jupyter/IRI-seq/script_from_github/Bead_bc_white_list/Spatial_R2_barcode_1.pickle,True


Preflight checks passed.


## 3. Prepare FASTQ names and sample sheets

The author scripts expect files named as `{sample}.R1.fastq.gz`, `{sample}.R2.fastq.gz`, and for the interaction pipeline `{sample}.R3.fastq.gz`.


In [3]:
def symlink_or_validate(src, dst, force=False):
    src = Path(src).resolve()
    dst = Path(dst)
    if not src.exists():
        raise FileNotFoundError(src)
    if dst.exists() or dst.is_symlink():
        try:
            same_target = dst.resolve() == src
        except FileNotFoundError:
            same_target = False
        if same_target:
            return 'exists'
        if not force:
            raise FileExistsError(f'{dst} already exists and does not point to {src}. Set FORCE_RELINK_FASTQ=True to replace it.')
        dst.unlink()
    os.symlink(src, dst)
    return 'linked'

prepared = []
prepared_map = {
    CDNA_SAMPLE: {
        'R1': FASTQ_SOURCE_FILES[CDNA_SAMPLE]['R1'],
        'R2': FASTQ_SOURCE_FILES[CDNA_SAMPLE]['R2'],
    },
    BEAD_SAMPLE: {
        'R1': FASTQ_SOURCE_FILES[BEAD_SAMPLE]['R1'],
        'R2': FASTQ_SOURCE_FILES[BEAD_SAMPLE]['R2'],
        'R3': FASTQ_SOURCE_FILES[BEAD_SAMPLE]['R3'],
    },
}

for sample, read_map in prepared_map.items():
    for read_name, src in read_map.items():
        dst = FASTQ_DIR / f'{sample}.{read_name}.fastq.gz'
        status = symlink_or_validate(src, dst, FORCE_RELINK_FASTQ)
        prepared.append({'sample': sample, 'read': read_name, 'source': str(src), 'prepared': str(dst), 'status': status})

CDNA_SAMPLE_FILE = SAMPLE_SHEET_DIR / 'cDNA_samples.txt'
BEAD_SAMPLE_FILE = SAMPLE_SHEET_DIR / 'bead_samples.txt'
CDNA_SAMPLE_FILE.write_text(CDNA_SAMPLE + '\n')
BEAD_SAMPLE_FILE.write_text(BEAD_SAMPLE + '\n')

display(pd.DataFrame(prepared))
print('cDNA sample sheet:', CDNA_SAMPLE_FILE)
print('bead sample sheet:', BEAD_SAMPLE_FILE)


,sample,read,source,prepared,status
0,20231221_FY1_3_cDNA,R1,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/data/SR...,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/...,linked
1,20231221_FY1_3_cDNA,R2,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/data/SR...,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/...,linked
2,20231221_FY1_3_connection,R1,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/data/SR...,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/...,linked
3,20231221_FY1_3_connection,R2,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/data/SR...,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/...,linked
4,20231221_FY1_3_connection,R3,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/data/SR...,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/...,linked


cDNA sample sheet: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/sample_sheets/cDNA_samples.txt
bead sample sheet: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/sample_sheets/bead_samples.txt


## 4. Import author scripts

The notebook imports the scripts directly from the GitHub clone. It does not edit those scripts; compatibility fixes are done in notebook state.


In [4]:
EASYSPATIAL_DIR = SCRIPT_FROM_GITHUB / 'EasySpatial'
BEAD_SCRIPT_DIR = SCRIPT_FROM_GITHUB / 'Bead_interaction_pipeline'
for path in [str(EASYSPATIAL_DIR), str(BEAD_SCRIPT_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

Spatial_UMI_barcode_extraction = importlib.import_module('Spatial_UMI_barcode_extraction')
Fastq_trim_multi_files = importlib.import_module('Fastq_trim_multi_files')
STAR_module = importlib.import_module('STAR')
Sam_filter_multi_files = importlib.import_module('Sam_filter_multi_files')
Sam_rm_dup_barcode_UMI_multi_files = importlib.import_module('Sam_rm_dup_barcode_UMI_multi_files')
Sam_gene_counting_multi_files = importlib.import_module('Sam_gene_counting_multi_files')
Summary_gene_count_multi_files = importlib.import_module('Summary_gene_count_multi_files')
Generate_adata = importlib.import_module('Generate_adata')
Count_reads = importlib.import_module('Count_reads')

Bead_UMI_barcode_extraction = importlib.import_module('UMI_barcode_extraction')
Bead_spatial_barcode_extraction = importlib.import_module('spatial_barcode_extraction')
Bead_Remove_duplicate_barcode = importlib.import_module('Remove_duplicate_barcode')

# Author bead spatial script reads this global variable internally.
Bead_spatial_barcode_extraction.list_barcode_4_file2 = str(BARCODE_4_FILE)

print('Imported author scripts from:', SCRIPT_FROM_GITHUB)


Imported author scripts from: /p1/zulab_users/jtian/my_jupyter/IRI-seq/script_from_github


## 5. cDNA FASTQ to expression matrix

Author order: barcode/UMI attach -> polyA trimming -> STAR alignment -> SAM filtering -> barcode+UMI duplicate removal -> gene counting -> count summary -> AnnData generation.


In [5]:
adata_full_path = ADATA_FOLDER / 'adata_full.h5ad'

if RUN_CDNA_PIPELINE:
    if adata_full_path.exists() and not FORCE_RERUN_CDNA:
        print('Skipping cDNA pipeline because output exists:', adata_full_path)
    else:
        Spatial_UMI_barcode_extraction.extract_spatial_barcode_files(
            str(FASTQ_DIR), str(CDNA_SAMPLE_FILE), str(FASTQ_BARCODE_ATTACHED), CORE_CDNA,
            str(BARCODE_1_FILE), str(BARCODE_2_FILE), str(BARCODE_3_FILE), str(BARCODE_4_BEAD1_FILE),
            mismatch_rate=1,
        )
        Fastq_trim_multi_files.Fastq_trim_files(
            str(FASTQ_BARCODE_ATTACHED), str(CDNA_SAMPLE_FILE), str(FASTQ_TRIMMED), CORE_CDNA,
            adapter_seq='AAAAAAAA', cutadapt_path=str(CUTADAPT),
        )
        STAR_module.Fastq_star_alignment_multi_files(
            str(FASTQ_TRIMMED), str(CDNA_SAMPLE_FILE), str(SAM_STAR), CORE_CDNA,
            index=str(STAR_INDEX), star_path=str(STAR),
        )
        Sam_filter_multi_files.Sam_filter_files(
            str(SAM_STAR), str(CDNA_SAMPLE_FILE), str(SAM_FILTERED), CORE_CDNA,
            samtools_path=str(SAMTOOLS),
        )
        Sam_rm_dup_barcode_UMI_multi_files.rm_dup_files(
            str(SAM_FILTERED), str(CDNA_SAMPLE_FILE), str(SAM_RMDUP), CORE_CDNA,
        )
        Sam_gene_counting_multi_files.scRNA_count_parallel(
            str(SAM_RMDUP), str(CDNA_SAMPLE_FILE), str(BED_GENE_COUNT), str(GENE_REFERENCE_PICKLE), CORE_CDNA,
        )
        Summary_gene_count_multi_files.Gene_count_summary(
            str(BED_GENE_COUNT), str(CDNA_SAMPLE_FILE), str(SUMMARY_GENE_COUNT),
        )
        Generate_adata.generate_adata_from_gene_count(
            str(SUMMARY_GENE_COUNT), str(BED_GENE_COUNT / 'gene_anno.csv'), str(ADATA_FOLDER), UMI_LIMIT_FOR_ADATA,
        )
        print('Saved:', adata_full_path)
else:
    print('RUN_CDNA_PIPELINE=False; cDNA FASTQ processing was skipped.')



    --------------------------start attaching UMI-----------------------------
    input folder: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/prepared_fastq
    sample ID: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/sample_sheets/cDNA_samples.txt
    output_folder: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/cDNA/Fastq_barcode_attached
    Barcode 1 file: /p1/zulab_users/jtian/my_jupyter/IRI-seq/script_from_github/Bead_bc_white_list/Spatial_R2_barcode_1.pickle
    Barcode 2 file: /p1/zulab_users/jtian/my_jupyter/IRI-seq/script_from_github/Bead_bc_white_list/Spatial_R2_barcode_2.pickle
    Barcode 3 file: /p1/zulab_users/jtian/my_jupyter/IRI-seq/script_from_github/Bead_bc_white_list/Spatial_R2_barcode_3.pickle
    Barcode 4 file: /p1/zulab_users/jtian/my_jupyter/IRI-seq/script_from_github/Bead_bc_white_list/Spatial_R2_barcode_4_bead1.pickle
    ___________________________________________________________________________
  

[bam_sort_core] merging from 0 files and 10 in-memory blocks...



All samples are processed.

    --------------------------start removing duplicates-----------------------------
    input folder: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/cDNA/Sam_filtered
    sample ID: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/sample_sheets/cDNA_samples.txt
    output_folder: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/cDNA/Sam_rmdup
    ___________________________________________________________________________
    

    ---------------------------------------------------------------------------
    sample ID: 20231221_FY1_3_cDNA
    Total input read number: 25180667
    Total unique read number: 12992450
    Duplication rate: 0.4840307446979065
    ___________________________________________________________________________
    
~~~~~~~~~~~~~~~Duplicate removal done~~~~~~~~~~~~~~~~~~
Load gene reference file....
Start processing all file....
Start read the input file: /p2/zulab/jtian/data/IRISe

### cDNA read-count report


In [6]:
cdna_report_path = REPORT_DIR / 'cDNA_read_number_summary.csv'
if adata_full_path.exists():
    cdna_report = pd.DataFrame(index=[CDNA_SAMPLE])
    try:
        cdna_report['STAR_input_reads'] = Count_reads.Fastq_count_reads_files(str(FASTQ_BARCODE_ATTACHED) + '/', str(CDNA_SAMPLE_FILE), core=CORE_CDNA)
    except Exception as exc:
        cdna_report['STAR_input_reads'] = np.nan
        print('Could not count barcode-attached FASTQ reads:', repr(exc))
    try:
        star_counts = Count_reads.Count_Align_STAR_files(str(SAM_STAR), str(CDNA_SAMPLE_FILE), core=CORE_CDNA)
        star_counts.columns = ['STAR_input', 'STAR_unique', 'STAR_multi'][:star_counts.shape[1]]
        cdna_report = cdna_report.join(star_counts, how='left')
    except Exception as exc:
        print('Could not parse STAR final logs:', repr(exc))
    try:
        cdna_report['filtered_sam_reads'] = Count_reads.SAM_count_reads_files(str(SAM_FILTERED), str(CDNA_SAMPLE_FILE), core=CORE_CDNA)
        cdna_report['dedup_sam_reads'] = Count_reads.SAM_count_reads_files(str(SAM_RMDUP), str(CDNA_SAMPLE_FILE), core=CORE_CDNA)
        cdna_report['gene_assigned_reads'] = Count_reads.count_mapped_reads_files(str(BED_GENE_COUNT), str(CDNA_SAMPLE_FILE), core=CORE_CDNA)
    except Exception as exc:
        print('Could not count SAM/gene-assigned reads:', repr(exc))
    cdna_report.to_csv(cdna_report_path)
    display(cdna_report)
    print('Saved:', cdna_report_path)
else:
    print('cDNA AnnData does not exist yet; run the cDNA pipeline first.')


,STAR_input_reads,STAR_input,STAR_unique,STAR_multi,filtered_sam_reads,dedup_sam_reads,gene_assigned_reads
20231221_FY1_3_cDNA,168997196,141470693,25180667,18266481,25180667,12992450,10688260


Saved: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/reports/cDNA_read_number_summary.csv


## 6. Metadata loading and barcode normalization

cDNA barcodes use hyphen separators; bead-interaction receiver barcodes are concatenated. The helper below normalizes both to a compact receiver barcode. For this GEO metadata file, FY1_3 FASTQ-derived barcodes match the metadata row-name/index prefix, so annotation is selected by `metadata.index` containing `SAMPLE_ID` rather than by `orig.ident`.


In [3]:
def strip_sample_prefix(value):
    s = str(value)
    if '.' in s:
        s = s.split('.', 1)[1]
    if '-' in s:
        head, tail = s.rsplit('-', 1)
        if tail.isdigit():
            s = head
    return s


def compact_barcode(value):
    return strip_sample_prefix(value).replace('-', '')


def dashed_barcode(value):
    return strip_sample_prefix(value)

metadata_all = pd.read_csv(PROCESSED_METADATA_CSV, index_col=0)
if 'orig.ident' not in metadata_all.columns:
    raise ValueError('metadata is missing required column orig.ident')

metadata_index_mask = metadata_all.index.astype(str).str.contains(SAMPLE_ID, regex=False)
metadata_orig_mask = metadata_all['orig.ident'].astype(str).eq(SAMPLE_ID)
metadata_fy = metadata_all[metadata_index_mask].copy()
metadata_match_source = f'index contains {SAMPLE_ID}'
if metadata_fy.empty:
    metadata_fy = metadata_all[metadata_orig_mask].copy()
    metadata_match_source = f'orig.ident == {SAMPLE_ID}'
if metadata_fy.empty:
    raise ValueError(f'No metadata rows found for {SAMPLE_ID}')

metadata_fy['metadata_cell_name'] = metadata_fy.index.astype(str)
metadata_fy['receiver_barcode_dash'] = metadata_fy.index.map(dashed_barcode)
metadata_fy['receiver_barcode'] = metadata_fy.index.map(compact_barcode)
metadata_fy = metadata_fy.drop_duplicates('receiver_barcode', keep='first')

required_meta_cols = ['Annotation', 'UMAP1_gene_Expression', 'UMAP2_gene_Expression', 'UMAP1_spatial', 'UMAP2_spatial']
missing_meta_cols = [col for col in required_meta_cols if col not in metadata_fy.columns]
if missing_meta_cols:
    raise ValueError('metadata is missing required columns: ' + ', '.join(missing_meta_cols))

print('FY1_3 metadata match source:', metadata_match_source)
print('metadata rows with index containing FY1_3:', int(metadata_index_mask.sum()))
print('metadata rows with orig.ident == FY1_3:', int(metadata_orig_mask.sum()))
print('metadata rows satisfying both:', int((metadata_index_mask & metadata_orig_mask).sum()))
print('FY1_3 metadata beads after barcode dedup:', metadata_fy.shape[0])
display(metadata_fy[['orig.ident', 'Annotation', 'receiver_barcode_dash', 'receiver_barcode']].head())


FY1_3 metadata match source: index contains FY1_3
metadata rows with index containing FY1_3: 3920
metadata rows with orig.ident == FY1_3: 3920
metadata rows satisfying both: 123
FY1_3 metadata beads after barcode dedup: 3920


,orig.ident,Annotation,receiver_barcode_dash,receiver_barcode
20231221_FY1_3_cDNA.ACACCGTA-AACAACCG-CGCCTTAA-AGCTTACG-2,FO1_2,Cortical_associated_area_section1,ACACCGTA-AACAACCG-CGCCTTAA-AGCTTACG,ACACCGTAAACAACCGCGCCTTAAAGCTTACG
20231221_FY1_3_cDNA.ACACCGTA-AACCAGGT-AACCTCTC-GTCACTGA-2,FY3_3,Cortex_2_section2,ACACCGTA-AACCAGGT-AACCTCTC-GTCACTGA,ACACCGTAAACCAGGTAACCTCTCGTCACTGA
20231221_FY1_3_cDNA.ACACCGTA-AACCAGGT-AATCGCCT-GACGTTAC-2,FY2_4,Hypotalamus_section3_4,ACACCGTA-AACCAGGT-AATCGCCT-GACGTTAC,ACACCGTAAACCAGGTAATCGCCTGACGTTAC
20231221_FY1_3_cDNA.ACACCGTA-AACCTCTC-AAGAGGCA-GGATCGAT-2,FO2_4,Hypotalamus_section3_4,ACACCGTA-AACCTCTC-AAGAGGCA-GGATCGAT,ACACCGTAAACCTCTCAAGAGGCAGGATCGAT
20231221_FY1_3_cDNA.ACACCGTA-AACCTCTC-CAGTCGAA-GACCTCTA-2,FY1_2,Cortical_associated_area_section1,ACACCGTA-AACCTCTC-CAGTCGAA-GACCTCTA,ACACCGTAAACCTCTCCAGTCGAAGACCTCTA


## 7. Expression UMAP from reproduced cDNA matrix

The expression workflow follows the author Fig. 2-style parameters in Python/Scanpy: UMI filter, normalization, HVG selection, scaling, PCA, neighbors, Leiden clustering, and UMAP.


In [4]:
if RUN_EXPRESSION_ANALYSIS:
    import scanpy as sc
    import scipy.sparse as sp

    if not adata_full_path.exists():
        raise FileNotFoundError(f'Missing reproduced AnnData: {adata_full_path}. Run the cDNA pipeline first.')

    adata = sc.read_h5ad(adata_full_path)
    adata.var_names_make_unique()

    raw_total_counts = np.asarray(adata.X.sum(axis=1)).ravel()
    raw_gene_counts = np.asarray((adata.X > 0).sum(axis=1)).ravel() if not sp.issparse(adata.X) else np.asarray(adata.X.getnnz(axis=1)).ravel()
    adata.obs['reproduced_total_counts'] = raw_total_counts
    adata.obs['reproduced_n_genes'] = raw_gene_counts
    adata.obs['receiver_barcode_dash'] = adata.obs_names.map(dashed_barcode)
    adata.obs['receiver_barcode'] = adata.obs_names.map(compact_barcode)

    annotation_map = metadata_fy.set_index('receiver_barcode')['Annotation']
    adata.obs['Annotation'] = adata.obs['receiver_barcode'].map(annotation_map).fillna('Unannotated')

    keep = adata.obs['reproduced_total_counts'].ge(EXPRESSION_MIN_COUNTS)
    adata_qc = adata[keep].copy()
    if adata_qc.n_obs < 3:
        raise ValueError(f'Only {adata_qc.n_obs} cells remain after nCount_RNA >= {EXPRESSION_MIN_COUNTS}')

    adata_qc.layers['counts'] = adata_qc.X.copy()
    sc.pp.normalize_total(adata_qc, target_sum=1e4)
    sc.pp.log1p(adata_qc)

    n_top = min(HVG_N_TOP_GENES, adata_qc.n_vars)
    try:
        sc.pp.highly_variable_genes(adata_qc, n_top_genes=n_top, flavor='seurat_v3', layer='counts')
    except Exception as exc:
        print('seurat_v3 HVG unavailable; falling back to scanpy seurat flavor:', repr(exc))
        sc.pp.highly_variable_genes(adata_qc, n_top_genes=n_top, flavor='seurat')

    sc.pp.scale(adata_qc, max_value=10)
    n_pcs = min(EXPRESSION_PCS, adata_qc.n_obs - 1, adata_qc.n_vars - 1)
    sc.tl.pca(adata_qc, n_comps=n_pcs, svd_solver='arpack')
    neighbor_pcs = min(EXPRESSION_NEIGHBOR_PCS, n_pcs)
    sc.pp.neighbors(adata_qc, n_pcs=neighbor_pcs, random_state=RECONSTRUCTION_RANDOM_STATE)
    try:
        sc.tl.leiden(adata_qc, resolution=2, key_added='leiden_r2', random_state=RECONSTRUCTION_RANDOM_STATE)
    except Exception as exc:
        print('Leiden clustering unavailable; storing unclustered labels:', repr(exc))
        adata_qc.obs['leiden_r2'] = 'unclustered'
    sc.tl.umap(adata_qc, min_dist=EXPRESSION_UMAP_MIN_DIST, random_state=RECONSTRUCTION_RANDOM_STATE)

    expression_h5ad = EXPRESSION_OUTPUT / 'FY1_3_reproduced_expression_scanpy_umap.h5ad'
    expression_table = EXPRESSION_OUTPUT / 'FY1_3_reproduced_expression_umap_by_annotation.csv.gz'
    adata_qc.write(expression_h5ad)
    expr_df = adata_qc.obs[['receiver_barcode', 'receiver_barcode_dash', 'Annotation', 'leiden_r2', 'reproduced_total_counts', 'reproduced_n_genes']].copy()
    expr_df['expression_UMAP1'] = adata_qc.obsm['X_umap'][:, 0]
    expr_df['expression_UMAP2'] = adata_qc.obsm['X_umap'][:, 1]
    expr_df.to_csv(expression_table, compression='gzip')

    print('cells before filtering:', adata.n_obs)
    print('cells after nCount_RNA >=', EXPRESSION_MIN_COUNTS, ':', adata_qc.n_obs)
    print('metadata annotations matched:', int(expr_df['Annotation'].ne('Unannotated').sum()))
    print('Saved:', expression_h5ad)
    print('Saved:', expression_table)
    display(expr_df.head())
else:
    print('RUN_EXPRESSION_ANALYSIS=False; expression UMAP was skipped.')


NameError: name 'adata_full_path' is not defined

## 8. Bead-interaction FASTQ to deduplicated receiver-sender connections

This follows the author's bead interaction scripts and preserves their file formats.


In [ ]:
spatial_rmdup_path = DEDUPLICATE_SPATIAL / f'{BEAD_SAMPLE}.spatial.csv.gz'


def repair_bead_umi_attach_fastq(path):
    """Repair author interaction UMI-attach output when UMI newline splits the header.

    Some runs of the author script write each retained read as five physical lines:
    @bc1,bc2,bc3,bc4,UMI\n + ,original_header\n + seq/+ /qual.
    The next author script expects a four-line FASTQ record, so merge those two
    header fragments without changing the barcode content.
    """
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp_repaired')
    repaired_records = 0
    standard_records = 0
    with gzip.open(path, 'rt') as src, gzip.open(tmp, 'wt', compresslevel=1) as dst:
        while True:
            line1 = src.readline()
            if not line1:
                break
            line2 = src.readline()
            if not line2:
                raise ValueError(f'Truncated UMI-attach FASTQ after header: {path}')
            if line1.startswith('@') and line2.startswith(','):
                seq = src.readline()
                plus = src.readline()
                qual = src.readline()
                if not qual:
                    raise ValueError(f'Truncated five-line UMI-attach record in {path}')
                dst.write(line1.rstrip('\n') + line2)
                dst.write(seq)
                dst.write(plus)
                dst.write(qual)
                repaired_records += 1
            else:
                seq = line2
                plus = src.readline()
                qual = src.readline()
                if not qual:
                    raise ValueError(f'Truncated four-line UMI-attach record in {path}')
                dst.write(line1)
                dst.write(seq)
                dst.write(plus)
                dst.write(qual)
                standard_records += 1
    os.replace(tmp, path)
    print(f'Repaired {path}: merged {repaired_records} five-line records; kept {standard_records} standard records.')


def validate_bead_umi_attach_fastq(path, n_records=1000):
    bad_headers = []
    with gzip.open(path, 'rt') as handle:
        for idx in range(n_records):
            header = handle.readline()
            if not header:
                break
            seq = handle.readline()
            plus = handle.readline()
            qual = handle.readline()
            if not (header.startswith('@') and plus.startswith('+') and len(header.strip().split(',')) >= 5):
                bad_headers.append((idx + 1, header.strip(), plus.strip()))
                break
    if bad_headers:
        raise ValueError(f'UMI-attach FASTQ header validation failed: {bad_headers[:1]}')


if RUN_BEAD_PIPELINE:
    if spatial_rmdup_path.exists() and not FORCE_RERUN_BEADS:
        print('Skipping bead pipeline because output exists:', spatial_rmdup_path)
    else:
        umi_attach_fastq = UMI_ATTACH / f'{BEAD_SAMPLE}.R2.fastq.gz'
        if umi_attach_fastq.exists() and not FORCE_RERUN_BEADS:
            print('Reusing existing UMI-attach FASTQ:', umi_attach_fastq)
        else:
            Bead_UMI_barcode_extraction.extract_spatial_barcode_files(
                str(FASTQ_DIR), str(BEAD_SAMPLE_FILE), str(UMI_ATTACH), CORE_BEADS,
                str(BARCODE_1_FILE), str(BARCODE_2_FILE), str(BARCODE_3_FILE), str(BARCODE_4_BEAD1_FILE),
                mismatch_rate=1,
            )
        repair_bead_umi_attach_fastq(umi_attach_fastq)
        validate_bead_umi_attach_fastq(umi_attach_fastq)

        Bead_spatial_barcode_extraction.list_barcode_4_file2 = str(BARCODE_4_FILE)
        Bead_spatial_barcode_extraction.extract_spatial_barcode_files(
            str(UMI_ATTACH), str(BEAD_SAMPLE_FILE), str(SPATIAL_BARCODE), CORE_BEADS,
            str(BARCODE_1_FILE), str(BARCODE_2_FILE), str(BARCODE_3_FILE), str(BARCODE_4_FILE),
        )
        Bead_Remove_duplicate_barcode.remove_duplicates_files(
            str(SPATIAL_BARCODE), str(BEAD_SAMPLE_FILE), str(DEDUPLICATE_SPATIAL), CORE_BEADS,
        )
        print('Saved:', spatial_rmdup_path)
else:
    print('RUN_BEAD_PIPELINE=False; bead FASTQ processing was skipped.')


Reusing existing UMI-attach FASTQ: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/beads/UMI_attach/20231221_FY1_3_connection.R2.fastq.gz
Repaired /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/beads/UMI_attach/20231221_FY1_3_connection.R2.fastq.gz: merged 0 five-line records; kept 49397157 standard records.

    --------------------------start attaching UMI-----------------------------
    input folder: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/beads/UMI_attach
    sample ID: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/sample_sheets/bead_samples.txt
    output_folder: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/beads/Spatial_barcode_extraction
    Barcode 1 file: /p1/zulab_users/jtian/my_jupyter/IRI-seq/script_from_github/Bead_bc_white_list/Spatial_R2_barcode_1.pickle
    Barcode 2 file: /p1/zulab_users/jtian/my_jupyter/IRI-seq/script_from_github/Bead_bc_white_list/Spatial_R2_barco

### Bead interaction read-count report


In [ ]:
def count_gzip_lines(path):
    with gzip.open(path, 'rt') as handle:
        return sum(1 for _ in handle)

if spatial_rmdup_path.exists():
    bead_report_rows = []
    raw_r2 = FASTQ_DIR / f'{BEAD_SAMPLE}.R2.fastq.gz'
    umi_r2 = UMI_ATTACH / f'{BEAD_SAMPLE}.R2.fastq.gz'
    spatial_txt = SPATIAL_BARCODE / f'{BEAD_SAMPLE}.spatial.txt.gz'
    bead_report_rows.append({
        'sample': BEAD_SAMPLE,
        'total_reads': count_gzip_lines(raw_r2) // 4 if raw_r2.exists() else np.nan,
        'Filtering_bead1_barcode': count_gzip_lines(umi_r2) // 4 if umi_r2.exists() else np.nan,
        'Filtering_bead2_barcode': count_gzip_lines(spatial_txt) if spatial_txt.exists() else np.nan,
        'Remove_duplicates_file_lines': count_gzip_lines(spatial_rmdup_path) if spatial_rmdup_path.exists() else np.nan,
    })
    bead_read_number = pd.DataFrame(bead_report_rows)
    bead_read_number_path = BEAD_REPORT_FOLDER / 'read_number.csv'
    bead_read_number.to_csv(bead_read_number_path, index=False)
    display(bead_read_number)
    print('Saved:', bead_read_number_path)
else:
    print('Deduplicated spatial connection file does not exist yet.')


,sample,total_reads,Filtering_bead1_barcode,Filtering_bead2_barcode,Remove_duplicates_file_lines
0,20231221_FY1_3_connection,56642169,49397157,44611326,31942236


Saved: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/beads/report/read_num_spatial_barcode/read_number.csv


## 9. Spatial reconstruction from bead-bead interactions

The reconstruction follows the author's notebook logic: receiver-sender UMI counts, filter low-quality edges, log transform, dimensionality reduction, and UMAP.


In [ ]:
def read_spatial_connections(path):
    path = Path(path)
    raw = pd.read_csv(path, compression='gzip', header=None, dtype=str)
    if raw.empty:
        return pd.DataFrame(columns=['Bead1_seq', 'UMI', 'Bead2_seq'])
    if raw.shape[1] == 1:
        parsed = raw.iloc[:, 0].str.split(',', n=2, expand=True)
    else:
        parsed = raw.iloc[:, :3].copy()
    parsed.columns = ['Bead1_seq', 'UMI', 'Bead2_seq']
    parsed = parsed.dropna()
    for column in parsed.columns:
        parsed[column] = parsed[column].astype(str).str.strip()
    parsed = parsed[~parsed['Bead1_seq'].isin(['Bead1_seq', 'receiver_barcode'])]
    parsed = parsed[(parsed['Bead1_seq'] != '') & (parsed['UMI'] != '') & (parsed['Bead2_seq'] != '')]
    return parsed


def build_connection_edges(connections, min_umis=1):
    edges = (
        connections.groupby(['Bead1_seq', 'Bead2_seq'], observed=True)
        .size()
        .reset_index(name='n_umi')
    )
    if min_umis > 1:
        edges = edges[edges['n_umi'] >= min_umis].copy()
    return edges

if RUN_RECONSTRUCTION:
    spatial_umap_path = CONNECTION_OUTPUT / f'{BEAD_SAMPLE}_spatial_reconstruction_umap.csv'
    edge_path = CONNECTION_OUTPUT / 'receiver_sender_edges_min_umi.csv.gz'
    matrix_path = CONNECTION_OUTPUT / 'receiver_sender_connection_matrix_min_umi.npz'
    if spatial_umap_path.exists() and not FORCE_RERUN_RECONSTRUCTION:
        print('Skipping reconstruction because output exists:', spatial_umap_path)
    else:
        import scipy.sparse as sp
        from sklearn.decomposition import PCA, TruncatedSVD
        from sklearn.preprocessing import StandardScaler
        import umap

        if not spatial_rmdup_path.exists():
            raise FileNotFoundError(f'Missing deduplicated bead connection file: {spatial_rmdup_path}')

        connections = read_spatial_connections(spatial_rmdup_path)
        edges = build_connection_edges(connections, min_umis=MIN_CONNECTION_UMIS)
        if edges.empty:
            raise ValueError(f'No receiver-sender edges remain after n_umi >= {MIN_CONNECTION_UMIS}')
        edges.to_csv(edge_path, index=False, compression='gzip')

        receiver_index = pd.Index(pd.unique(edges['Bead1_seq']), name='receiver_barcode')
        sender_index = pd.Index(pd.unique(edges['Bead2_seq']), name='sender_barcode')
        receiver_codes = receiver_index.get_indexer(edges['Bead1_seq'])
        sender_codes = sender_index.get_indexer(edges['Bead2_seq'])
        connection_matrix = sp.csr_matrix(
            (edges['n_umi'].astype(np.float32), (receiver_codes, sender_codes)),
            shape=(len(receiver_index), len(sender_index)),
        )
        sp.save_npz(matrix_path, connection_matrix)
        receiver_index.to_frame(index=False).to_csv(CONNECTION_OUTPUT / 'receiver_barcodes.csv', index=False)
        sender_index.to_frame(index=False).to_csv(CONNECTION_OUTPUT / 'sender_barcodes.csv', index=False)

        matrix_cells = connection_matrix.shape[0] * connection_matrix.shape[1]
        max_valid_components = max(2, min(connection_matrix.shape[0] - 1, connection_matrix.shape[1] - 1))
        n_components = min(RECONSTRUCTION_COMPONENTS, max_valid_components)
        matrix_log = connection_matrix.copy().astype(np.float32)
        matrix_log.data = np.log1p(matrix_log.data)

        if RECONSTRUCTION_MODE == 'dense' or (RECONSTRUCTION_MODE == 'auto' and matrix_cells <= DENSE_MATRIX_MAX_CELLS):
            dense = matrix_log.toarray()
            dense_scaled = StandardScaler().fit_transform(dense)
            reducer = PCA(n_components=n_components, random_state=RECONSTRUCTION_RANDOM_STATE)
            reduced = reducer.fit_transform(dense_scaled)
            explained = getattr(reducer, 'explained_variance_ratio_', np.repeat(np.nan, n_components))
            reduction_used = 'dense_PCA_StandardScaler'
        else:
            reducer = TruncatedSVD(n_components=n_components, random_state=RECONSTRUCTION_RANDOM_STATE)
            reduced = reducer.fit_transform(matrix_log)
            explained = getattr(reducer, 'explained_variance_ratio_', np.repeat(np.nan, n_components))
            reduction_used = 'sparse_TruncatedSVD_log1p'

        explained_df = pd.DataFrame({
            'component': np.arange(1, len(explained) + 1),
            'explained_variance_ratio': explained,
            'cumulative_explained_variance': np.cumsum(explained),
            'reduction_used': reduction_used,
        })
        explained_path = CONNECTION_OUTPUT / 'connection_reduction_explained_variance.csv'
        explained_df.to_csv(explained_path, index=False)

        neighbors = min(RECONSTRUCTION_UMAP_N_NEIGHBORS, max(2, reduced.shape[0] - 1))
        mapper = umap.UMAP(
            n_neighbors=neighbors,
            min_dist=RECONSTRUCTION_UMAP_MIN_DIST,
            spread=RECONSTRUCTION_UMAP_SPREAD,
            metric='euclidean',
            random_state=RECONSTRUCTION_RANDOM_STATE,
            n_epochs=RECONSTRUCTION_UMAP_EPOCHS,
            verbose=True,
        )
        coords = mapper.fit_transform(reduced)
        spatial_umap = pd.DataFrame({
            'receiver_barcode': receiver_index.astype(str),
            'spatial_UMAP1': coords[:, 0],
            'spatial_UMAP2': coords[:, 1],
        })
        spatial_umap.to_csv(spatial_umap_path, index=False)

        print('raw unique molecule rows:', connections.shape[0])
        print('filtered receiver-sender edges:', edges.shape[0])
        print('connection matrix shape:', connection_matrix.shape)
        print('reduction:', reduction_used)
        print('Saved:', edge_path)
        print('Saved:', matrix_path)
        print('Saved:', explained_path)
        print('Saved:', spatial_umap_path)
        display(spatial_umap.head())
else:
    print('RUN_RECONSTRUCTION=False; spatial reconstruction was skipped.')


/p1/zulab_users/jtian/anaconda3/envs/my_IRISeq_py38/lib/python3.8/site-packages/umap/umap_.py:1945: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


UMAP(min_dist=0.23, n_epochs=500, n_jobs=1, n_neighbors=19, random_state=42, spread=0.5, verbose=True)
Sat Jul  4 08:00:45 2026 Construct fuzzy simplicial set
Sat Jul  4 08:00:45 2026 Finding Nearest Neighbors
Sat Jul  4 08:00:45 2026 Building RP forest with 10 trees
Sat Jul  4 08:00:52 2026 NN descent for 13 iterations
	 1  /  13
	 2  /  13
	 3  /  13
	 4  /  13
	Stopping threshold met -- exiting after 4 iterations
Sat Jul  4 08:01:04 2026 Finished Nearest Neighbor Search
Sat Jul  4 08:01:05 2026 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Sat Jul  4 08:01:18 2026 Finished embedding
raw unique molecule rows: 31942236
filtered receiver-sender edges: 225728
connection matrix shape: (10700, 5449)
reduction: sparse_TruncatedSVD_log1p
Saved: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/connections/receiver_sender_edges_min_umi.csv.gz
Saved: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/connections/receiver_sender_connection_matrix_min_umi.npz
Saved: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/connections/connection_reduction_explained_variance.csv
Saved: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/connections/20231221_FY1_3_connection_spatia

,receiver_barcode,spatial_UMAP1,spatial_UMAP2
0,ACACCGTAAACAACCGCGCCTTAAAGCTTACG,7.344245,6.177719
1,ACACCGTAAACAACCGCGTGCATATCGCTAGA,5.461193,6.528493
2,ACACCGTAAACAACCGTTAACGCGGATCAGCA,6.610504,2.588394
3,ACACCGTAAACCAGGTAACCTCTCGTCACTGA,7.341220,1.679593
4,ACACCGTAAACCAGGTAATCGCCTGACGTTAC,6.677181,7.981693


## 10. Author-style main figures

The two requested main outputs are the reproduced expression UMAP and reproduced bead-bead spatial reconstruction UMAP, both colored by GEO `Annotation`.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl


def ordered_labels(values):
    labels = pd.Series(values).fillna('Unannotated').astype(str)
    return sorted(labels.unique())


def make_color_map(labels):
    try:
        import scanpy as sc
        palette = list(sc.pl.palettes.default_102)
    except Exception:
        palette = []
    if len(palette) < len(labels):
        cmaps = ['tab20', 'tab20b', 'tab20c', 'Set3', 'Dark2', 'Paired']
        for cmap_name in cmaps:
            cmap = mpl.colormaps[cmap_name]
            palette.extend([mpl.colors.to_hex(cmap(i)) for i in np.linspace(0, 1, cmap.N)])
    return {label: palette[i % len(palette)] for i, label in enumerate(labels)}


def scatter_by_label(ax, df, x, y, label_col, color_map, point_size=3, alpha=0.85):
    for label, sub in df.groupby(label_col, sort=True):
        ax.scatter(sub[x], sub[y], s=point_size, linewidths=0, alpha=alpha, color=color_map.get(str(label), '#999999'), label=str(label))
    ax.set_aspect('equal', adjustable='datalim')

expr_table = EXPRESSION_OUTPUT / 'FY1_3_reproduced_expression_umap_by_annotation.csv.gz'
spatial_umap_path = CONNECTION_OUTPUT / f'{BEAD_SAMPLE}_spatial_reconstruction_umap.csv'
if not expr_table.exists():
    raise FileNotFoundError(f'Missing expression UMAP table: {expr_table}')
if not spatial_umap_path.exists():
    raise FileNotFoundError(f'Missing spatial reconstruction UMAP table: {spatial_umap_path}')

expr_df = pd.read_csv(expr_table, index_col=0)
expr_df['Annotation'] = expr_df['Annotation'].fillna('Unannotated').astype(str)

spatial_df = pd.read_csv(spatial_umap_path)
spatial_df['receiver_barcode'] = spatial_df['receiver_barcode'].astype(str)
spatial_df = spatial_df.merge(
    metadata_fy[['receiver_barcode', 'receiver_barcode_dash', 'Annotation']],
    on='receiver_barcode', how='left',
)
spatial_df['Annotation'] = spatial_df['Annotation'].fillna('Unannotated').astype(str)
spatial_annotated_path = CONNECTION_OUTPUT / f'{BEAD_SAMPLE}_spatial_reconstruction_umap_by_annotation.csv.gz'
spatial_df.to_csv(spatial_annotated_path, index=False, compression='gzip')

labels = sorted(set(ordered_labels(expr_df['Annotation'])) | set(ordered_labels(spatial_df['Annotation'])))
color_map = make_color_map(labels)

fig, ax = plt.subplots(figsize=(8, 7))
scatter_by_label(ax, expr_df, 'expression_UMAP1', 'expression_UMAP2', 'Annotation', color_map, point_size=3)
ax.set_title('FY1_3 gene-expression UMAP')
ax.set_xlabel('UMAP1 (gene expression)')
ax.set_ylabel('UMAP2 (gene expression)')
handles, handle_labels = ax.get_legend_handles_labels()
ax.legend(handles, handle_labels, title='Annotation', loc='center left', bbox_to_anchor=(1.02, 0.5), markerscale=2, frameon=False)
expr_png = FIGURE_OUTPUT / 'FY1_3_expression_umap_by_annotation.png'
expr_pdf = FIGURE_OUTPUT / 'FY1_3_expression_umap_by_annotation.pdf'
fig.savefig(expr_png, dpi=300, bbox_inches='tight')
fig.savefig(expr_pdf, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 7))
scatter_by_label(ax, spatial_df, 'spatial_UMAP1', 'spatial_UMAP2', 'Annotation', color_map, point_size=3)
ax.set_title('FY1_3 bead-bead spatial reconstruction')
ax.set_xlabel('UMAP1 (bead-bead interaction)')
ax.set_ylabel('UMAP2 (bead-bead interaction)')
handles, handle_labels = ax.get_legend_handles_labels()
ax.legend(handles, handle_labels, title='Annotation', loc='center left', bbox_to_anchor=(1.02, 0.5), markerscale=2, frameon=False)
spatial_png = FIGURE_OUTPUT / 'FY1_3_spatial_reconstruction_by_annotation.png'
spatial_pdf = FIGURE_OUTPUT / 'FY1_3_spatial_reconstruction_by_annotation.pdf'
fig.savefig(spatial_png, dpi=300, bbox_inches='tight')
fig.savefig(spatial_pdf, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
scatter_by_label(axes[0], expr_df, 'expression_UMAP1', 'expression_UMAP2', 'Annotation', color_map, point_size=2.5)
axes[0].set_title('Gene-expression UMAP')
axes[0].set_xlabel('UMAP1 (gene expression)')
axes[0].set_ylabel('UMAP2 (gene expression)')
scatter_by_label(axes[1], spatial_df, 'spatial_UMAP1', 'spatial_UMAP2', 'Annotation', color_map, point_size=2.5)
axes[1].set_title('Spatial reconstruction')
axes[1].set_xlabel('UMAP1 (bead-bead interaction)')
axes[1].set_ylabel('UMAP2 (bead-bead interaction)')
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map[label], markersize=6, label=label) for label in labels]
fig.legend(handles=handles, title='Annotation', loc='center left', bbox_to_anchor=(1.01, 0.5), frameon=False)
two_panel_png = FIGURE_OUTPUT / 'FY1_3_expression_and_spatial_two_panel.png'
two_panel_pdf = FIGURE_OUTPUT / 'FY1_3_expression_and_spatial_two_panel.pdf'
fig.savefig(two_panel_png, dpi=300, bbox_inches='tight')
fig.savefig(two_panel_pdf, bbox_inches='tight')
plt.show()

print('Saved:', expr_png)
print('Saved:', expr_pdf)
print('Saved:', spatial_png)
print('Saved:', spatial_pdf)
print('Saved:', two_panel_png)
print('Saved:', two_panel_pdf)
print('Saved annotated spatial table:', spatial_annotated_path)


## 11. Validation against GEO processed files

This section is deliberately separate from the main FASTQ-derived analysis. It checks whether reproduced outputs agree with the author-provided processed files.


In [ ]:
def safe_corr(x, y, method='pearson'):
    x = pd.Series(x, dtype='float64')
    y = pd.Series(y, dtype='float64')
    mask = x.notna() & y.notna()
    if mask.sum() < 3:
        return np.nan
    return x[mask].corr(y[mask], method=method)


def validation_metric(metrics, name, value):
    if isinstance(value, (np.integer, np.floating)):
        value = value.item()
    metrics[name] = value

validation_metrics = {}

if RUN_VALIDATION:
    import scipy.sparse as sp
    from scipy.io import mmread
    from scipy.spatial import procrustes
    from sklearn.neighbors import NearestNeighbors

    # cDNA validation against the series-level processed expression matrix.
    if adata_full_path.exists():
        import scanpy as sc
        reproduced = sc.read_h5ad(adata_full_path)
        reproduced.obs['receiver_barcode'] = reproduced.obs_names.map(compact_barcode)
        reproduced_totals = pd.Series(np.asarray(reproduced.X.sum(axis=1)).ravel(), index=reproduced.obs['receiver_barcode'])
        reproduced_genes = pd.Index(reproduced.var_names.astype(str))

        processed_barcodes = pd.read_csv(PROCESSED_BARCODES_TSV, sep='\t', header=None)[0].astype(str)
        processed_genes_df = pd.read_csv(PROCESSED_GENES_TSV, sep='\t', header=None)
        processed_gene_names = processed_genes_df.iloc[:, -1].astype(str)
        processed_compact = processed_barcodes.map(compact_barcode)
        fy_compact_set = set(metadata_fy['receiver_barcode'])
        fy_col_mask = processed_compact.isin(fy_compact_set).to_numpy()

        validation_metric(validation_metrics, 'processed_total_barcodes', int(processed_barcodes.shape[0]))
        validation_metric(validation_metrics, 'processed_FY1_3_barcodes_from_metadata', int(fy_col_mask.sum()))
        validation_metric(validation_metrics, 'reproduced_cDNA_barcodes', int(reproduced.n_obs))
        validation_metric(validation_metrics, 'reproduced_cDNA_genes', int(reproduced.n_vars))
        validation_metric(validation_metrics, 'processed_reproduced_gene_name_overlap', int(len(set(reproduced_genes) & set(processed_gene_names))))

        if fy_col_mask.sum() > 0:
            processed_mtx = mmread(PROCESSED_COUNT_MTX).tocsr()
            processed_fy = processed_mtx[:, fy_col_mask]
            processed_totals = pd.Series(
                np.asarray(processed_fy.sum(axis=0)).ravel(),
                index=processed_compact[fy_col_mask].to_numpy(),
            )
            common = reproduced_totals.index.intersection(processed_totals.index)
            validation_metric(validation_metrics, 'cDNA_common_barcodes_for_total_UMI', int(len(common)))
            validation_metric(validation_metrics, 'cDNA_total_UMI_pearson', safe_corr(reproduced_totals.loc[common], processed_totals.loc[common], 'pearson'))
            validation_metric(validation_metrics, 'cDNA_total_UMI_spearman', safe_corr(reproduced_totals.loc[common], processed_totals.loc[common], 'spearman'))
            pd.DataFrame({
                'receiver_barcode': common,
                'reproduced_total_UMI': reproduced_totals.loc[common].to_numpy(),
                'processed_total_UMI': processed_totals.loc[common].to_numpy(),
            }).to_csv(VALIDATION_OUTPUT / 'cDNA_total_UMI_reproduced_vs_processed.csv', index=False)
    else:
        validation_metric(validation_metrics, 'cDNA_validation_status', 'skipped_missing_reproduced_adata')

    # Interaction validation against the FY1_3 processed connection file.
    if spatial_rmdup_path.exists() and PROCESSED_CONNECTION_CSV.exists():
        reproduced_conn = read_spatial_connections(spatial_rmdup_path)
        processed_conn = read_spatial_connections(PROCESSED_CONNECTION_CSV)
        reproduced_edges_all = build_connection_edges(reproduced_conn, min_umis=1)
        processed_edges_all = build_connection_edges(processed_conn, min_umis=1)
        reproduced_edges_f = reproduced_edges_all[reproduced_edges_all['n_umi'] >= MIN_CONNECTION_UMIS].copy()
        processed_edges_f = processed_edges_all[processed_edges_all['n_umi'] >= MIN_CONNECTION_UMIS].copy()

        rep_set = set(zip(reproduced_edges_f['Bead1_seq'], reproduced_edges_f['Bead2_seq']))
        proc_set = set(zip(processed_edges_f['Bead1_seq'], processed_edges_f['Bead2_seq']))
        union = rep_set | proc_set
        inter = rep_set & proc_set
        validation_metric(validation_metrics, 'reproduced_connection_unique_molecules', int(reproduced_conn.shape[0]))
        validation_metric(validation_metrics, 'processed_connection_unique_molecules', int(processed_conn.shape[0]))
        validation_metric(validation_metrics, 'reproduced_edges_min_umi', int(reproduced_edges_f.shape[0]))
        validation_metric(validation_metrics, 'processed_edges_min_umi', int(processed_edges_f.shape[0]))
        validation_metric(validation_metrics, 'interaction_edge_jaccard_min_umi', float(len(inter) / len(union)) if union else np.nan)

        rep_edge_series = reproduced_edges_all.set_index(['Bead1_seq', 'Bead2_seq'])['n_umi']
        proc_edge_series = processed_edges_all.set_index(['Bead1_seq', 'Bead2_seq'])['n_umi']
        common_edges = rep_edge_series.index.intersection(proc_edge_series.index)
        validation_metric(validation_metrics, 'interaction_common_edges_all', int(len(common_edges)))
        validation_metric(validation_metrics, 'interaction_edge_UMI_pearson', safe_corr(rep_edge_series.loc[common_edges], proc_edge_series.loc[common_edges], 'pearson'))
        validation_metric(validation_metrics, 'interaction_edge_UMI_spearman', safe_corr(rep_edge_series.loc[common_edges], proc_edge_series.loc[common_edges], 'spearman'))
        pd.DataFrame({
            'Bead1_seq': [idx[0] for idx in common_edges],
            'Bead2_seq': [idx[1] for idx in common_edges],
            'reproduced_n_umi': rep_edge_series.loc[common_edges].to_numpy(),
            'processed_n_umi': proc_edge_series.loc[common_edges].to_numpy(),
        }).to_csv(VALIDATION_OUTPUT / 'interaction_edge_UMI_reproduced_vs_processed.csv', index=False)
    else:
        validation_metric(validation_metrics, 'interaction_validation_status', 'skipped_missing_connection_file')

    # Spatial coordinate validation against GEO metadata coordinates.
    spatial_umap_path = CONNECTION_OUTPUT / f'{BEAD_SAMPLE}_spatial_reconstruction_umap.csv'
    if spatial_umap_path.exists():
        spatial_df = pd.read_csv(spatial_umap_path)
        spatial_df['receiver_barcode'] = spatial_df['receiver_barcode'].astype(str)
        ref_coords = metadata_fy[['receiver_barcode', 'UMAP1_spatial', 'UMAP2_spatial', 'Annotation']].copy()
        compare = spatial_df.merge(ref_coords, on='receiver_barcode', how='inner')
        compare = compare.dropna(subset=['spatial_UMAP1', 'spatial_UMAP2', 'UMAP1_spatial', 'UMAP2_spatial'])
        validation_metric(validation_metrics, 'spatial_common_barcodes_for_coordinate_validation', int(compare.shape[0]))
        if compare.shape[0] >= 3:
            ref = compare[['UMAP1_spatial', 'UMAP2_spatial']].astype(float).to_numpy()
            rep = compare[['spatial_UMAP1', 'spatial_UMAP2']].astype(float).to_numpy()
            ref_proc, rep_proc, disparity = procrustes(ref, rep)
            validation_metric(validation_metrics, 'spatial_procrustes_disparity', float(disparity))
            validation_metric(validation_metrics, 'spatial_procrustes_axis1_pearson', safe_corr(ref_proc[:, 0], rep_proc[:, 0], 'pearson'))
            validation_metric(validation_metrics, 'spatial_procrustes_axis2_pearson', safe_corr(ref_proc[:, 1], rep_proc[:, 1], 'pearson'))

            k = min(10, compare.shape[0] - 1)
            if k >= 2:
                ref_nn = NearestNeighbors(n_neighbors=k + 1).fit(ref).kneighbors(return_distance=False)[:, 1:]
                rep_nn = NearestNeighbors(n_neighbors=k + 1).fit(rep).kneighbors(return_distance=False)[:, 1:]
                overlap = [len(set(a) & set(b)) / k for a, b in zip(ref_nn, rep_nn)]
                validation_metric(validation_metrics, 'spatial_knn10_mean_overlap', float(np.mean(overlap)))

            compare_out = compare.copy()
            compare_out[['reference_procrustes_1', 'reference_procrustes_2']] = ref_proc
            compare_out[['reproduced_procrustes_1', 'reproduced_procrustes_2']] = rep_proc
            compare_out.to_csv(VALIDATION_OUTPUT / 'spatial_coordinates_reproduced_vs_metadata.csv', index=False)
    else:
        validation_metric(validation_metrics, 'spatial_validation_status', 'skipped_missing_reconstructed_coordinates')

    validation_json = VALIDATION_OUTPUT / 'validation_summary.json'
    validation_md = VALIDATION_OUTPUT / 'validation_summary.md'
    with open(validation_json, 'w') as handle:
        json.dump(validation_metrics, handle, indent=2)

    validation_rows = pd.DataFrame({'metric': list(validation_metrics.keys()), 'value': list(validation_metrics.values())})
    markdown_lines = ['# FY1_3 validation summary', '', '| metric | value |', '|---|---|']
    for _, row in validation_rows.iterrows():
        metric = str(row['metric']).replace('|', '\\|')
        value = str(row['value']).replace('|', '\\|')
        markdown_lines.append(f'| {metric} | {value} |')
    markdown_lines.append('')
    validation_md.write_text('\n'.join(markdown_lines))

    display(validation_rows)
    print('Saved:', validation_json)
    print('Saved:', validation_md)
else:
    print('RUN_VALIDATION=False; validation section was skipped.')


## 12. Output manifest

Run this at the end to collect the main generated files.


In [ ]:
manifest_targets = [
    ADATA_FOLDER / 'adata_full.h5ad',
    EXPRESSION_OUTPUT / 'FY1_3_reproduced_expression_scanpy_umap.h5ad',
    EXPRESSION_OUTPUT / 'FY1_3_reproduced_expression_umap_by_annotation.csv.gz',
    DEDUPLICATE_SPATIAL / f'{BEAD_SAMPLE}.spatial.csv.gz',
    CONNECTION_OUTPUT / 'receiver_sender_edges_min_umi.csv.gz',
    CONNECTION_OUTPUT / 'receiver_sender_connection_matrix_min_umi.npz',
    CONNECTION_OUTPUT / f'{BEAD_SAMPLE}_spatial_reconstruction_umap.csv',
    FIGURE_OUTPUT / 'FY1_3_expression_umap_by_annotation.png',
    FIGURE_OUTPUT / 'FY1_3_spatial_reconstruction_by_annotation.png',
    FIGURE_OUTPUT / 'FY1_3_expression_and_spatial_two_panel.png',
    VALIDATION_OUTPUT / 'validation_summary.json',
    VALIDATION_OUTPUT / 'validation_summary.md',
]
manifest = []
for path in manifest_targets:
    path = Path(path)
    manifest.append({
        'path': str(path),
        'exists': path.exists(),
        'size_MB': round(path.stat().st_size / 1024**2, 3) if path.exists() else np.nan,
    })
manifest_df = pd.DataFrame(manifest)
manifest_path = OUTPUT_DIR / 'output_manifest.csv'
manifest_df.to_csv(manifest_path, index=False)
display(manifest_df)
print('Saved:', manifest_path)


,path,exists,size_MB
0,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/...,True,59.318
1,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/...,True,672.173
2,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/...,True,0.193
3,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/...,True,551.790
4,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/...,True,2.335
5,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/...,True,0.640
6,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/...,True,0.546
7,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/...,True,0.527
8,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/...,True,0.783
9,/p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/...,True,0.937


Saved: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/output/demo-FY1_3-pipeline/output_manifest.csv


In [4]:
# Inspect metadata rows whose barcode/index name contains FY1_3.
metadata_debug_path = Path('/p2/zulab/jtian/data/IRISeq/demo_FY1_3/data/GSE270383_meta_data.csv.gz')
metadata_debug = pd.read_csv(metadata_debug_path, index_col=0)

fy1_3_index_rows = metadata_debug[metadata_debug.index.astype(str).str.contains('FY1_3', regex=False)].copy()
fy1_3_index_rows['barcode_index'] = fy1_3_index_rows.index.astype(str)

print('metadata file:', metadata_debug_path)
print('total metadata rows:', metadata_debug.shape[0])
print('rows with barcode/index containing FY1_3:', fy1_3_index_rows.shape[0])
print('rows with orig.ident == FY1_3:', int(metadata_debug['orig.ident'].astype(str).eq('FY1_3').sum()))
print('rows satisfying both:', int((metadata_debug.index.astype(str).str.contains('FY1_3', regex=False) & metadata_debug['orig.ident'].astype(str).eq('FY1_3')).sum()))

print('orig.ident distribution among barcode/index contains FY1_3:')
print(fy1_3_index_rows['orig.ident'].astype(str).value_counts().to_string())

print('Annotation distribution among barcode/index contains FY1_3:')
print(fy1_3_index_rows['Annotation'].astype(str).value_counts().to_string())

cols_to_show = ['barcode_index', 'orig.ident', 'Annotation']
extra_cols = [col for col in ['UMAP1_gene_Expression', 'UMAP2_gene_Expression', 'UMAP1_spatial', 'UMAP2_spatial'] if col in fy1_3_index_rows.columns]
display(fy1_3_index_rows[cols_to_show + extra_cols])

metadata file: /p2/zulab/jtian/data/IRISeq/demo_FY1_3/data/GSE270383_meta_data.csv.gz
total metadata rows: 128405
rows with barcode/index containing FY1_3: 3920
rows with orig.ident == FY1_3: 3920
rows satisfying both: 123
orig.ident distribution among barcode/index contains FY1_3:
orig.ident
FY2_5    276
FY3_2    231
FY2_4    224
FO1_4    206
FY3_4    194
FO3_2    190
FO2_2    187
FY1_5    180
FY2_2    173
FO3_5    165
FO1_2    164
FO1_5    163
FY3_5    162
FY2_3    150
FO1_3    146
FO3_4    137
FO3_3    136
FO2_5    131
FY3_3    124
FO2_4    124
FY1_3    123
FY1_2    121
FY1_4    119
FO2_3     94
Annotation distribution among barcode/index contains FY1_3:
Annotation
Cortical_associated_area_section1        897
Cortex_2_section3_4                      272
Thalamus_Central_Ventral_section3_4      262
Cortex_3_section3_4                      248
Amygdala_section3_4                      237
Hypotalamus_section3_4                   168
Cortex_1_section3_4                      159
Cortex_2

,barcode_index,orig.ident,Annotation,UMAP1_gene_Expression,UMAP2_gene_Expression,UMAP1_spatial,UMAP2_spatial
20231221_FY1_3_cDNA.ACACCGTA-AACAACCG-CGCCTTAA-AGCTTACG-2,20231221_FY1_3_cDNA.ACACCGTA-AACAACCG-CGCCTTAA-AGCTTACG-2,FO1_2,Cortical_associated_area_section1,0.161425,-3.071127,6.712849,4.047127
20231221_FY1_3_cDNA.ACACCGTA-AACCAGGT-AACCTCTC-GTCACTGA-2,20231221_FY1_3_cDNA.ACACCGTA-AACCAGGT-AACCTCTC-GTCACTGA-2,FY3_3,Cortex_2_section2,-4.959306,0.404128,10.110597,4.006552
20231221_FY1_3_cDNA.ACACCGTA-AACCAGGT-AATCGCCT-GACGTTAC-2,20231221_FY1_3_cDNA.ACACCGTA-AACCAGGT-AATCGCCT-GACGTTAC-2,FY2_4,Hypotalamus_section3_4,-1.847576,2.989066,-6.097342,-0.380492
20231221_FY1_3_cDNA.ACACCGTA-AACCTCTC-AAGAGGCA-GGATCGAT-2,20231221_FY1_3_cDNA.ACACCGTA-AACCTCTC-AAGAGGCA-GGATCGAT-2,FO2_4,Hypotalamus_section3_4,-2.013855,3.174358,5.105602,10.567973
20231221_FY1_3_cDNA.ACACCGTA-AACCTCTC-CAGTCGAA-GACCTCTA-2,20231221_FY1_3_cDNA.ACACCGTA-AACCTCTC-CAGTCGAA-GACCTCTA-2,FY1_2,Cortical_associated_area_section1,1.335242,-1.993857,2.965482,-0.604000
...,...,...,...,...,...,...,...
20231221_FY1_3_cDNA.TTTCCGCTT-GGTCTACA-CCATACTG-ACGCTCTA-2,20231221_FY1_3_cDNA.TTTCCGCTT-GGTCTACA-CCATACTG-ACGCTCTA-2,FY1_5,Hippocampus_CA1_CA2_CA3_section3_4,-0.617323,-7.842427,5.095501,9.905439
20231221_FY1_3_cDNA.TTTCCGCTT-TCCGTAGT-CGTGCATA-AGGCTGAT-2,20231221_FY1_3_cDNA.TTTCCGCTT-TCCGTAGT-CGTGCATA-AGGCTGAT-2,FO3_3,Caudate Putamen_section2,8.688209,1.605655,1.805735,5.321919
20231221_FY1_3_cDNA.TTTCCGCTT-TTAACGCG-CGTAGGTT-TACGGCTC-2,20231221_FY1_3_cDNA.TTTCCGCTT-TTAACGCG-CGTAGGTT-TACGGCTC-2,FY3_5,Amygdala_section3_4,0.678610,1.108171,11.475423,6.465095
20231221_FY1_3_cDNA.TTTCCGCTT-TTCTGAGG-AGGTCCTA-GTCCGTTA-2,20231221_FY1_3_cDNA.TTTCCGCTT-TTCTGAGG-AGGTCCTA-GTCCGTTA-2,FY3_4,Hippocampus_CA1_CA2_CA3_section3_4,2.874533,-2.634185,2.315045,7.221277


In [5]:
metadata_debug

,orig.ident,nCount_RNA,nFeature_RNA,Annotation,UMAP1_gene_Expression,UMAP2_gene_Expression,UMAP1_spatial,UMAP2_spatial
20231221_FY1_3_cDNA.ACACCGTA-AACAACCG-CGCCTTAA-AGCTTACG-2,FO1_2,11568,4017,Cortical_associated_area_section1,0.161425,-3.071127,6.712849,4.047127
20231221_FY1_3_cDNA.ACACCGTA-AACCAGGT-AACCTCTC-GTCACTGA-2,FY3_3,12835,3521,Cortex_2_section2,-4.959306,0.404128,10.110597,4.006552
20231221_FY1_3_cDNA.ACACCGTA-AACCAGGT-AATCGCCT-GACGTTAC-2,FY2_4,11249,3782,Hypotalamus_section3_4,-1.847576,2.989066,-6.097342,-0.380492
20231221_FY1_3_cDNA.ACACCGTA-AACCTCTC-AAGAGGCA-GGATCGAT-2,FO2_4,1546,676,Hypotalamus_section3_4,-2.013855,3.174358,5.105602,10.567973
20231221_FY1_3_cDNA.ACACCGTA-AACCTCTC-CAGTCGAA-GACCTCTA-2,FY1_2,1142,718,Cortical_associated_area_section1,1.335242,-1.993857,2.965482,-0.604000
...,...,...,...,...,...,...,...,...
20231221_FO3_5_cDNA.TTTCCGCTT-TGAGGATG-CACTTCCA-CGCTAATC-29,FY2_3,1661,711,Nucleus Accumbens_section2,6.643631,-1.763657,0.880680,-0.220933
20231221_FO3_5_cDNA.TTTCCGCTT-TTCCGCTT-CAGTAAGG-TACACGAC-29,FY3_4,2370,905,Meninges_section3_4,-4.146035,-6.236059,10.150673,7.616868
20231221_FO3_5_cDNA.TTTCCGCTT-TTCCGCTT-GTCAGAAC-TACGTCGT-29,FY2_4,15143,4375,Cortex_3_section3_4,2.292793,-1.195941,4.048572,-2.638043
20231221_FO3_5_cDNA.TTTCCGCTT-TTCTGAGG-ATATCCGG-AGGCTGAT-29,FO1_5,1011,615,Interneurons_section3_4,-1.956018,-7.114633,8.757072,3.764089
